# SIEM event classifier — Qwen3-4B fine-tune on the merged datasets

**Runtime:** GPU (a T4 is enough). Run the cells top to bottom.

1 install · 2 config · 3 helpers · 4 download merged datasets from Drive · 5 load · 6 clean · 7 split + save · 8 train · 9 evaluate · 10 try it · 11 (optional) final model on all data

If Colab disconnects mid-training: re-run cells 1–3, set `RESUME = True` in cell 2, then re-run cell 8.
After any restart you can jump straight to cells 9 / 10 after 1–3 (they load the saved adapter from Drive).

In [ ]:
# ===== 1. INSTALL =====
!pip install -q unsloth

In [ ]:
# ===== 2. CONFIG =====
from unsloth import FastLanguageModel          # import Unsloth first so it can patch transformers / trl

import os, re, io, gc, json, glob, math, shutil, warnings
import numpy as np
import pandas as pd
from google.colab import drive, auth
warnings.filterwarnings("ignore", category=FutureWarning)
drive.mount("/content/drive")

# ---- data source ----
DRIVE_FOLDER_ID = "1BfnEaKcx402MH6_pbvxn5xo2btQ66DVk"   # merged-datasets folder
LOCAL_DIR = ""        # optional: a mounted path instead of the folder ID, e.g. "/content/drive/MyDrive/SIEM_ASTRA/merged"
RAW_DIR   = "/content/merged_raw"                       # local copy of the downloaded files

# ---- outputs (on Drive, so they survive disconnects) ----
BASE     = "/content/drive/MyDrive/SIEM_ASTRA"
DATA_DIR = f"{BASE}/data_qwen3"
CKPT     = f"{BASE}/ckpt_qwen3"
OUT      = f"{BASE}/adapters/qwen3_4b_siem"
OUT_ALL  = f"{BASE}/adapters/qwen3_4b_siem_all"
for p in (DATA_DIR, CKPT, OUT):
    os.makedirs(p, exist_ok=True)

# ---- model / training ----
MODEL_NAME  = "unsloth/Qwen3-4B-Instruct-2507"   # non-thinking Qwen3-4B; "unsloth/Qwen3-4B" (hybrid thinking) also works
MAX_SEQ_LEN = 512
LORA_R      = 16
EPOCHS      = 1
MAX_STEPS   = -1      # > 0 caps the run (overrides EPOCHS) if the merged set is very large
EVAL_STEPS  = 100     # evaluate + checkpoint every N steps
LR          = 2e-4
SEED        = 42
RESUME      = False   # True = continue from the latest checkpoint in CKPT after a disconnect

# ---- cleaning ----
MIN_PER_CLASS   = 20     # rare event types are oversampled in TRAIN (only) up to this many rows
LABEL_PURITY    = 0.90   # if >= 90% of an event type carries one label, its other-label rows are dropped as noise
URL_MAX, UA_MAX = 300, 200

# Fields the model sees (same set as v1). Leaky fields (tags, raw_log, tampered_file, mitre_*, severity,
# attack_vector) and lab-specific fields (IPs, timestamps, event_id) are never shown to the model.
FEATURES   = ["log_source", "dest_port", "method", "url", "status_code", "bytes", "user", "auth_method", "user_agent"]
LEAKY_DROP = ["tags", "raw_log", "tampered_file"]

SYSTEM_PROMPT = ("You are a SOC analyst. Classify the security event. "
                 'Reply with JSON only: {"label": 0 for benign or 1 for malicious, "event_type": "<snake_case type>"}')

def to_text(r):
    return (f"log_source={r['log_source']}; port={r['dest_port']}; method={r['method']}; url={r['url']}; "
            f"status={r['status_code']}; bytes={r['bytes']}; user={r['user']}; auth={r['auth_method']}; ua={r['user_agent']}")

def prompt_messages(r):
    return [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "Analyse this event: " + to_text(r)}]

def pair(r):
    answer = json.dumps({"label": int(r["label"]), "event_type": r["event_type"]})
    return {"messages": prompt_messages(r) + [{"role": "assistant", "content": answer}]}

print("config ready")

In [ ]:
# ===== 3. HELPERS (reading, cleaning, training, inference) =====
import torch
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import get_chat_template, train_on_responses_only
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

DATA_EXT  = (".json", ".jsonl", ".ndjson", ".csv", ".tsv", ".xlsx", ".xls", ".parquet")
SKIP_FILE = re.compile(r"^(train|val|test)(_full)?\.json$|^clean_events\.json$|^test_predictions\.csv$", re.I)
NULLS     = {"", "nan", "none", "null", "nil", "-", "n/a", "na", "<na>", "undefined"}
LABEL_MAP = {"1": 1, "true": 1, "yes": 1, "malicious": 1, "attack": 1, "anomaly": 1, "bad": 1,
             "0": 0, "false": 0, "no": 0, "benign": 0, "normal": 0, "clean": 0, "good": 0}
ALIASES = {   # other column names seen in SIEM exports -> our schema
    "src_ip": "source_ip", "srcip": "source_ip", "src_addr": "source_ip", "client_ip": "source_ip",
    "dst_ip": "dest_ip", "dstip": "dest_ip", "destination_ip": "dest_ip", "dst_addr": "dest_ip",
    "src_port": "source_port", "sport": "source_port",
    "dst_port": "dest_port", "dport": "dest_port", "destination_port": "dest_port", "port": "dest_port",
    "http_method": "method", "request_method": "method",
    "uri": "url", "path": "url", "request_uri": "url", "url_path": "url",
    "status": "status_code", "http_status": "status_code", "response_code": "status_code", "statuscode": "status_code",
    "bytes_sent": "bytes", "response_bytes": "bytes", "body_bytes_sent": "bytes", "size": "bytes",
    "useragent": "user_agent", "http_user_agent": "user_agent", "ua": "user_agent",
    "username": "user", "user_name": "user", "account": "user",
    "auth": "auth_method", "authentication_method": "auth_method",
    "event": "event_type", "eventtype": "event_type", "event_name": "event_type",
    "is_malicious": "label", "malicious": "label", "is_attack": "label",
    "logsource": "log_source",
    "time": "timestamp", "datetime": "timestamp", "event_time": "timestamp",
}

# ---------- reading ----------
def norm_col(c):
    c = re.sub(r"[^0-9a-z]+", "_", str(c).strip().lower()).strip("_")
    return ALIASES.get(c, c)

def coalesce_dupes(df):
    """Merge columns that ended up with the same name (e.g. src_ip + source_ip); first non-null wins."""
    if not df.columns.duplicated().any():
        return df
    out = {}
    for c in dict.fromkeys(df.columns):
        block = df.loc[:, df.columns == c]
        out[c] = block.bfill(axis=1).iloc[:, 0] if block.shape[1] > 1 else block.iloc[:, 0]
    return pd.DataFrame(out, index=df.index)

def _json_records(text):
    try:
        data = json.loads(text)
    except json.JSONDecodeError:                      # JSON Lines / NDJSON
        data, bad = [], 0
        for line in text.splitlines():
            line = line.strip().rstrip(",")
            if line in ("", "[", "]"):
                continue
            try:
                data.append(json.loads(line))
            except json.JSONDecodeError:
                bad += 1
        if bad:
            print(f"    {bad} unreadable lines skipped")
    if isinstance(data, dict):
        data = next((v for v in data.values() if isinstance(v, list)), [data])
    return [x for x in data if isinstance(x, dict)]

def read_any(path):
    ext = os.path.splitext(path)[1].lower()
    if ext in (".csv", ".tsv"):
        return pd.read_csv(path, sep="\t" if ext == ".tsv" else ",", low_memory=False, encoding_errors="replace")
    if ext in (".xlsx", ".xls"):
        return pd.concat(pd.read_excel(path, sheet_name=None).values(), ignore_index=True)
    if ext == ".parquet":
        return pd.read_parquet(path)
    with open(path, encoding="utf-8-sig", errors="replace") as f:
        return pd.json_normalize(_json_records(f.read().strip()))

# ---------- cleaning ----------
def norm_label(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return np.nan
    s = str(v).strip().lower()
    s = s[:-2] if s.endswith(".0") else s
    return LABEL_MAP.get(s, np.nan)

def clean_str(s, maxlen=None, case=None):
    s = s.astype("string").str.strip().str.replace(r"\s+", " ", regex=True)
    s = s.mask(s.isna() | s.str.lower().isin(NULLS).fillna(False).astype(bool))
    if case == "lower":
        s = s.str.lower()
    elif case == "upper":
        s = s.str.upper()
    s = s.fillna("none").astype(str)
    return s.str.slice(0, maxlen) if maxlen else s

def clean_num(s, lo=0, hi=None):
    raw = s.astype(str)
    n = pd.to_numeric(raw, errors="coerce")
    n = n.fillna(pd.to_numeric(raw.str.extract(r"(\d+(?:\.\d+)?)", expand=False), errors="coerce"))  # "80/tcp" -> 80
    ok = n.notna() & np.isfinite(n) & (n >= lo)
    if hi is not None:
        ok &= n <= hi
    n = n.where(ok).round()
    return n.map(lambda x: "none" if pd.isna(x) else str(int(x))).astype(object)

def prepare_features(df):
    """Normalise column names and the model-input fields. Used for training data AND for new events."""
    df = df.copy()
    df.columns = [norm_col(c) for c in df.columns]
    df = coalesce_dupes(df)
    for c in FEATURES:
        if c not in df.columns:
            df[c] = pd.NA
    df["dest_port"]   = clean_num(df["dest_port"], 0, 65535)
    df["status_code"] = clean_num(df["status_code"], 100, 599)
    df["bytes"]       = clean_num(df["bytes"], 0)
    df["log_source"]  = clean_str(df["log_source"], 40, "lower").str.replace(r"[^0-9a-z]+", "_", regex=True)
    df["method"]      = clean_str(df["method"], 16, "upper")
    df["url"]         = clean_str(df["url"], URL_MAX)
    df["user"]        = clean_str(df["user"], 64)
    df["auth_method"] = clean_str(df["auth_method"], 32, "lower")
    df["user_agent"]  = clean_str(df["user_agent"], UA_MAX)
    return df

# ---------- training ----------
def load_lora_model(name=MODEL_NAME):
    model, tok = FastLanguageModel.from_pretrained(name, max_seq_length=MAX_SEQ_LEN, load_in_4bit=True)
    model = FastLanguageModel.get_peft_model(
        model, r=LORA_R, lora_alpha=LORA_R, lora_dropout=0, bias="none",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        use_gradient_checkpointing="unsloth", random_state=SEED)
    if not getattr(tok, "chat_template", None):
        tok = get_chat_template(tok, chat_template="qwen3-instruct")
    tok.padding_side = "right"
    return model, tok

def chat_text(tok, messages, gen_prompt=False):
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=gen_prompt, enable_thinking=False)

def build_text_dataset(tok, files):
    ds = load_dataset("json", data_files={"x": files})["x"]
    ds = ds.map(lambda b: {"text": [chat_text(tok, m) for m in b["messages"]]},
                batched=True, remove_columns=ds.column_names)
    n0 = len(ds)
    ds = ds.filter(lambda b: [len(x) <= MAX_SEQ_LEN for x in tok(b["text"], add_special_tokens=False)["input_ids"]],
                   batched=True)
    if len(ds) < n0:
        print(f"  dropped {n0 - len(ds)} examples longer than {MAX_SEQ_LEN} tokens")
    return ds

def make_trainer(model, tok, train_ds, eval_ds, out_dir):
    has_eval = eval_ds is not None
    args = SFTConfig(
        dataset_text_field="text", max_seq_length=MAX_SEQ_LEN,
        per_device_train_batch_size=8, per_device_eval_batch_size=16, gradient_accumulation_steps=2,
        num_train_epochs=EPOCHS, max_steps=MAX_STEPS, learning_rate=LR, warmup_steps=10,
        lr_scheduler_type="linear", weight_decay=0.01, optim="adamw_8bit",
        fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
        logging_steps=25, eval_strategy="steps" if has_eval else "no", eval_steps=EVAL_STEPS,
        save_strategy="steps", save_steps=EVAL_STEPS if has_eval else 200, save_total_limit=2,
        output_dir=out_dir, report_to="none", seed=SEED)
    trainer = SFTTrainer(model=model, tokenizer=tok, train_dataset=train_ds, eval_dataset=eval_ds, args=args)
    return train_on_responses_only(trainer, instruction_part="<|im_start|>user\n",
                                   response_part="<|im_start|>assistant\n")

def latest_ckpt(d):
    cks = glob.glob(f"{d}/checkpoint-*")
    return max(cks, key=lambda p: int(p.rsplit("-", 1)[1])) if cks else None

def show_masking(trainer, tok):
    try:
        ids = [t for t in trainer.train_dataset[0]["labels"] if t != -100]
        print("loss is computed only on ->", repr(tok.decode(ids)))
    except Exception as e:
        print("(mask check skipped:", e, ")")

# ---------- inference ----------
_THINK = re.compile(r"<think>.*?</think>", re.S)
_JSON  = re.compile(r"\{.*?\}", re.S)

def parse_pred(txt):
    txt = _THINK.sub("", txt or "").strip()
    m = _JSON.search(txt)
    try:
        j = json.loads(m.group(0))
        return int(j["label"]), str(j["event_type"]), True
    except Exception:
        return -1, "INVALID", False

def ensure_inference_model(path=OUT):
    global model, tok
    if "model" not in globals():
        model, tok = FastLanguageModel.from_pretrained(path, max_seq_length=MAX_SEQ_LEN, load_in_4bit=True)
    FastLanguageModel.for_inference(model)
    tok.padding_side = "left"

@torch.no_grad()
def generate(prompts, bs=32, max_new_tokens=48):
    outs = []
    for i in range(0, len(prompts), bs):
        enc = tok(prompts[i:i + bs], return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True,
                             pad_token_id=tok.pad_token_id)
        outs += tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return outs

print("helpers ready")

In [ ]:
# ===== 4. DOWNLOAD THE MERGED DATASETS FROM DRIVE =====
if LOCAL_DIR:
    SRC_DIR = LOCAL_DIR
    print("using mounted folder:", SRC_DIR)
else:
    auth.authenticate_user()                      # approve the popup -> Drive API access
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload
    svc = build("drive", "v3", cache_discovery=False)
    FOLDER, SHORTCUT, SHEET = ("application/vnd.google-apps.folder",
                               "application/vnd.google-apps.shortcut",
                               "application/vnd.google-apps.spreadsheet")

    def list_children(fid):
        items, token = [], None
        while True:
            r = svc.files().list(q=f"'{fid}' in parents and trashed=false", pageSize=1000, pageToken=token,
                                 fields="nextPageToken, files(id,name,mimeType,size,shortcutDetails)",
                                 supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
            items += r.get("files", [])
            token = r.get("nextPageToken")
            if not token:
                return items

    def fetch(file_id, path, export_mime=None):
        req = (svc.files().export_media(fileId=file_id, mimeType=export_mime) if export_mime
               else svc.files().get_media(fileId=file_id, supportsAllDrives=True))
        with io.FileIO(path, "wb") as fh:
            dl, done = MediaIoBaseDownload(fh, req, chunksize=50 * 1024 * 1024), False
            while not done:
                _, done = dl.next_chunk()

    def download_folder(fid, dest, depth=0):
        os.makedirs(dest, exist_ok=True)
        for f in list_children(fid):
            fid2, mime, name = f["id"], f["mimeType"], f["name"].replace("/", "_")
            if mime == SHORTCUT:
                fid2, mime = f["shortcutDetails"]["targetId"], f["shortcutDetails"]["targetMimeType"]
            if mime == FOLDER:
                download_folder(fid2, os.path.join(dest, name), depth + 1)
                continue
            path = os.path.join(dest, name + (".csv" if mime == SHEET else ""))
            if not path.lower().endswith(DATA_EXT):
                print("  skip (not a data file):", name)
                continue
            if os.path.exists(path):
                stem, ext = os.path.splitext(path)
                path = f"{stem}__{fid2[:6]}{ext}"
            fetch(fid2, path, "text/csv" if mime == SHEET else None)
            print(f"  {'  ' * depth}{name}  ({os.path.getsize(path) / 1e6:.2f} MB)")

    shutil.rmtree(RAW_DIR, ignore_errors=True)
    print("downloading folder", DRIVE_FOLDER_ID)
    download_folder(DRIVE_FOLDER_ID, RAW_DIR)
    SRC_DIR = RAW_DIR

files = sorted(p for p in glob.glob(f"{SRC_DIR}/**/*", recursive=True)
               if os.path.isfile(p) and p.lower().endswith(DATA_EXT))
print(f"\n{len(files)} data files found")
assert files, ("No data files found. Check the folder ID / that this Google account can open it, "
               "or add the folder to My Drive and set LOCAL_DIR to its /content/drive/... path.")

In [ ]:
# ===== 5. LOAD EVERY FILE =====
frames = {}
for path in files:
    name = os.path.relpath(path, SRC_DIR)
    if SKIP_FILE.match(os.path.basename(path)):
        print("skip (output of an earlier run):", name)
        continue
    try:
        df = read_any(path)
    except Exception as e:
        print(f"FAILED {name}: {e}")
        continue
    if df.empty or "messages" in df.columns:
        print("skip (empty or already chat-formatted):", name)
        continue
    df["member"] = df["member"].fillna(name) if "member" in df.columns else name
    frames[name] = df
    lab_col = next((c for c in df.columns if norm_col(c) == "label"), None)
    lab = df[lab_col].value_counts(dropna=False).to_dict() if lab_col else "no label column"
    print(f"{name}: {df.shape[0]} rows x {df.shape[1]} cols | label counts: {lab}")

assert frames, "nothing loaded"
raw = pd.concat(frames.values(), ignore_index=True, sort=False)
print(f"\nTOTAL raw rows: {len(raw)} | columns ({len(raw.columns)}):")
print(sorted(map(str, raw.columns)))

In [ ]:
# ===== 6. CLEAN =====
df = raw.copy()
df.columns = [norm_col(c) for c in df.columns]
df = coalesce_dupes(df)
missing = {"event_type", "label"} - set(df.columns)
assert not missing, f"required column(s) {missing} not found - add the real column name to ALIASES in cell 3"
log = {"raw rows": len(df)}

# 1) exact duplicate rows (same file twice, overlapping merges)
dup = df.drop(columns=["member"], errors="ignore").astype(str).duplicated()
df = df[~dup].copy()
log["exact duplicate rows removed"] = int(dup.sum())

# 2) label -> 0 / 1
df["label"] = df["label"].map(norm_label)
bad = df["label"].isna()
df = df[~bad].copy()
df["label"] = df["label"].astype(int)
log["unreadable label removed"] = int(bad.sum())

# 3) event_type -> snake_case
et = (df["event_type"].astype(str).str.strip().str.lower()
      .str.replace(r"[^0-9a-z]+", "_", regex=True).str.strip("_"))
bad = et.isna() | et.isin(NULLS)
df["event_type"] = et
df = df[~bad].copy()
log["missing event_type removed"] = int(bad.sum())

# 4) timestamps (kept for reference only - never shown to the model)
if "timestamp" in df.columns:
    ts = pd.to_datetime(df["timestamp"], format="mixed", utc=True, errors="coerce")
    log["unparseable timestamps (rows kept)"] = int(ts.isna().sum())
    df["timestamp"] = ts.dt.tz_convert("Asia/Kolkata")

# 5) model-input fields
df = prepare_features(df)
empty = (df[FEATURES[1:]] == "none").all(axis=1)
df = df[~empty].copy()
log["rows with no usable fields removed"] = int(empty.sum())

# 6) label noise inside an event_type
g = df.groupby("event_type")["label"]
maj = g.agg(lambda s: s.value_counts().idxmax())
purity = g.agg(lambda s: s.value_counts(normalize=True).max())
mixed = purity[purity < 1]
if len(mixed):
    print("event types that carry BOTH labels:")
    sub = df[df["event_type"].isin(mixed.index)]
    print(pd.crosstab(sub["event_type"], sub["label"]).to_string(), "\n")
fix = mixed[mixed >= LABEL_PURITY].index
noise = df["event_type"].isin(fix) & (df["label"] != df["event_type"].map(maj))
df = df[~noise].copy()
log[f"minority-label rows removed (types >= {LABEL_PURITY:.0%} pure)"] = int(noise.sum())

# 7) identical input with different labels can't be learned -> drop; then one row per unique input
types_before = set(df["event_type"])
df["_key"] = pd.util.hash_pandas_object(df[FEATURES], index=False).values
conflict = df.groupby("_key")["label"].transform("nunique") > 1
if conflict.any():
    print("excluded - an identical input also appears with the other label:")
    print(df.loc[conflict, "event_type"].value_counts().to_string(), "\n")
log["conflicting-input rows removed"] = int(conflict.sum())
d = df[~conflict].copy()
d["_n"] = d.groupby(["_key", "event_type"])["event_type"].transform("size")   # keep the majority event_type per input
d = (d.sort_values(["_key", "_n"], ascending=[True, False], kind="stable")
       .drop_duplicates("_key").drop(columns=["_key", "_n"]).reset_index(drop=True))
log["duplicate inputs collapsed"] = int((~conflict).sum() - len(d))
gone = sorted(types_before - set(d["event_type"]))
if gone:
    print("WARNING - event types with no rows left (the model will never predict these):", gone, "\n")

d = d.drop(columns=[c for c in LEAKY_DROP if c in d.columns])
log["FINAL rows"] = len(d)

print(pd.Series(log).to_string())
print("\nlabel balance:\n", d["label"].value_counts().rename({0: "benign", 1: "malicious"}).to_string())
print("\nevent types:\n", d["event_type"].value_counts().to_string())
if "member" in d.columns:
    print("\nper source file:\n", pd.crosstab(d["member"], d["label"]).to_string())
d.to_json(f"{DATA_DIR}/clean_events.json", orient="records", date_format="iso")
print("\nsaved", f"{DATA_DIR}/clean_events.json")

In [ ]:
# ===== 7. SPLIT 80 / 10 / 10 (stratified by event_type) + SAVE =====
from sklearn.model_selection import train_test_split

if "d" not in globals():
    d = pd.read_json(f"{DATA_DIR}/clean_events.json", dtype=False)

def strat_split(x, test_size):
    s = x["event_type"]
    s = s.where(s.map(s.value_counts()) >= 2, "__rare__")
    if (s == "__rare__").sum() == 1:
        s = s.replace("__rare__", s.value_counts().idxmax())
    try:
        return train_test_split(x, test_size=test_size, stratify=s, random_state=SEED)
    except ValueError as e:
        print("  (plain random split used:", e, ")")
        return train_test_split(x, test_size=test_size, random_state=SEED)

train, rest = strat_split(d, 0.2)
val, test = strat_split(rest, 0.5)

# oversample rare event types in TRAIN only (val / test stay untouched)
extra = [train[train["event_type"] == t].sample(MIN_PER_CLASS - n, replace=True, random_state=SEED)
         for t, n in train["event_type"].value_counts().items() if n < MIN_PER_CLASS]
train_os = pd.concat([train, *extra]).sample(frac=1, random_state=SEED).reset_index(drop=True)
print(f"train {len(train)} (+{len(train_os) - len(train)} oversampled) | val {len(val)} | test {len(test)}")
unseen = sorted((set(val["event_type"]) | set(test["event_type"])) - set(train["event_type"]))
if unseen:
    print("event types in val/test but not in train:", unseen)

for name, part, full in (("train", train_os, train), ("val", val, val), ("test", test, test)):
    with open(f"{DATA_DIR}/{name}.json", "w") as f:
        json.dump([pair(r) for r in part.to_dict("records")], f)
    full.to_json(f"{DATA_DIR}/{name}_full.json", orient="records", date_format="iso")

print(json.dumps(pair(train.iloc[0]), indent=2))
print(sorted(os.listdir(DATA_DIR)))

In [ ]:
# ===== 8. FINE-TUNE QWEN3-4B (LoRA, 4-bit) =====
model, tok = load_lora_model()
train_ds = build_text_dataset(tok, [f"{DATA_DIR}/train.json"])
val_ds   = build_text_dataset(tok, [f"{DATA_DIR}/val.json"])
print(f"train {len(train_ds)} | val {len(val_ds)}")
print(train_ds[0]["text"])
steps = MAX_STEPS if MAX_STEPS > 0 else math.ceil(len(train_ds) / 16) * EPOCHS
print(f"~{steps} optimizer steps (batch 8 x grad-accum 2)")

trainer = make_trainer(model, tok, train_ds, val_ds, CKPT)
show_masking(trainer, tok)
trainer.train(resume_from_checkpoint=latest_ckpt(CKPT) if RESUME else None)

model.save_pretrained(OUT)
tok.save_pretrained(OUT)
types = sorted(pd.read_json(f"{DATA_DIR}/train_full.json", dtype=False)["event_type"].unique())
with open(f"{OUT}/siem_config.json", "w") as f:
    json.dump({"base_model": MODEL_NAME, "features": FEATURES, "system_prompt": SYSTEM_PROMPT,
               "max_seq_len": MAX_SEQ_LEN, "event_types": types}, f, indent=2)

hist = pd.DataFrame(trainer.state.log_history)
cols = [c for c in ("loss", "eval_loss") if c in hist.columns]
print(hist.groupby("step")[cols].first().dropna(how="all").to_string())
print("saved to", OUT)

In [ ]:
# ===== 9. EVALUATE ON THE HELD-OUT TEST SET =====
from sklearn.metrics import classification_report, confusion_matrix
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

ensure_inference_model(OUT)
test_pairs = json.load(open(f"{DATA_DIR}/test.json"))
test_full  = pd.read_json(f"{DATA_DIR}/test_full.json", dtype=False)
prompts = [chat_text(tok, p["messages"][:-1], gen_prompt=True) for p in test_pairs]
truth   = [json.loads(p["messages"][-1]["content"]) for p in test_pairs]
outs = generate(prompts)

r = pd.DataFrame([parse_pred(o) for o in outs], columns=["p", "pet", "ok"])
r["y"]  = [t["label"] for t in truth]
r["et"] = [t["event_type"] for t in truth]
r["member"] = test_full["member"].values if "member" in test_full.columns else "all"
r["input"]  = [p["messages"][1]["content"] for p in test_pairs]
r["output"] = outs
r["label_hit"] = r["y"] == r["p"]
r["type_hit"]  = r["et"] == r["pet"]

print("invalid JSON:", int((~r["ok"]).sum()), "of", len(r))
v = r[r["ok"]]
print(classification_report(v["y"], v["p"], labels=[0, 1], target_names=["benign", "malicious"], digits=3, zero_division=0))
print("confusion matrix [rows = true benign/malicious]:\n", confusion_matrix(v["y"], v["p"], labels=[0, 1]))
print("\nevent_type accuracy:", round(r["type_hit"].mean(), 3))

print("\nper event type:")
print(r.groupby("et")[["label_hit", "type_hit"]].mean().round(2)
        .assign(n=r.groupby("et").size()).sort_values("type_hit").to_string())
print("\nper source file:")
print(r.groupby("member")[["label_hit", "type_hit"]].mean().round(3)
        .assign(n=r.groupby("member").size()).to_string())
print("\nmost common event_type confusions:")
print(r[~r["type_hit"]].groupby(["et", "pet"]).size().sort_values(ascending=False).head(15).to_string())

r.to_csv(f"{DATA_DIR}/test_predictions.csv", index=False)
print("\npredictions saved to", f"{DATA_DIR}/test_predictions.csv")

In [ ]:
# ===== 10. TRY IT ON NEW EVENTS =====
ensure_inference_model(OUT)

def analyse(events):
    """events: one dict or a list of dicts in the raw SIEM schema (extra / missing fields are fine)."""
    if isinstance(events, dict):
        events = [events]
    feats = prepare_features(pd.DataFrame(events))
    prompts = [chat_text(tok, prompt_messages(r), gen_prompt=True) for r in feats.to_dict("records")]
    rows = []
    for o in generate(prompts):
        label, et, ok = parse_pred(o)
        rows.append({"verdict": {1: "malicious", 0: "benign"}.get(label, "unparsed"), "label": label, "event_type": et})
    return pd.DataFrame(rows)

analyse([
    {"log_source": "web_server", "dest_port": 80, "method": "GET", "url": "/index.html", "status_code": 200,
     "bytes": 5120, "user_agent": "Mozilla/5.0 (X11; Linux x86_64) Firefox/128.0"},
    {"log_source": "web_server", "dest_port": 80, "method": "GET", "url": "/file?path=../../../etc/passwd",
     "status_code": 404, "bytes": 345, "user_agent": "curl/8.5.0"},
    {"log_source": "web_server", "dest_port": 80, "method": "GET",
     "url": "/login?id=1%27%20OR%20%271%27=%271", "status_code": 403, "bytes": 136, "user_agent": "sqlmap/1.8"},
    {"log_source": "system", "dest_port": 22, "user": "oracle", "auth_method": "password"},
])

In [ ]:
# ===== 11. (OPTIONAL) FINAL MODEL ON ALL CLEAN DATA =====
# Run only after you're happy with the test scores in cell 9. Cleanest way: Runtime -> Restart session,
# run cells 2 and 3, set RUN_FULL_DATA = True, then run this cell.
RUN_FULL_DATA = False

if RUN_FULL_DATA:
    for name in ("trainer", "model"):
        globals().pop(name, None)
    gc.collect()
    torch.cuda.empty_cache()

    model, tok = load_lora_model()
    all_ds = build_text_dataset(tok, [f"{DATA_DIR}/{s}.json" for s in ("train", "val", "test")])
    print("training events:", len(all_ds))
    trainer = make_trainer(model, tok, all_ds, None, f"{CKPT}_all")
    trainer.train(resume_from_checkpoint=latest_ckpt(f"{CKPT}_all") if RESUME else None)

    model.save_pretrained(OUT_ALL)
    tok.save_pretrained(OUT_ALL)
    if os.path.exists(f"{OUT}/siem_config.json"):
        shutil.copy(f"{OUT}/siem_config.json", f"{OUT_ALL}/siem_config.json")
    print("saved to", OUT_ALL)